# Stage 1: Domain Adaptation Training

Notebook ini untuk melatih IndoNanoT5 pada domain Python (Stage 1).

**Prerequisites:** Jalankan `01_setup_and_validation.ipynb` terlebih dahulu.

**Expected Results:**
- Training loss: ~3.0 → ~1.5
- Validation loss: ~2.8 → ~1.8
- Output model: `indot5-python-domain`
- Training time: ~2-3 jam pada T4 GPU

## 1. Setup Environment

In [ ]:
# Install dependencies
!pip install -q transformers>=4.35.0 peft>=0.7.0 datasets>=2.15.0 accelerate>=0.25.0
!pip install -q evaluate rouge_score bert_score
print('✓ Dependencies installed')

In [ ]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')
print('✓ Google Drive mounted')

In [ ]:
import os, sys

# Set project root - adjust path sesuai lokasi project di Drive
PROJECT_ROOT = '/content/drive/MyDrive/AQG'  # UBAH SESUAI PATH ANDA
os.chdir(PROJECT_ROOT)
sys.path.insert(0, PROJECT_ROOT)

print(f'Working directory: {os.getcwd()}')

In [ ]:
import torch

# Check GPU
if not torch.cuda.is_available():
    raise RuntimeError('GPU not available! Go to Runtime > Change runtime type > T4 GPU')

gpu_name = torch.cuda.get_device_name(0)
gpu_memory = torch.cuda.get_device_properties(0).total_memory / 1e9
print(f'✓ GPU: {gpu_name}')
print(f'✓ Memory: {gpu_memory:.1f} GB')

## 2. Load Domain Dataset

In [ ]:
from src.finetuned.data.dataset_loader import DatasetLoader

loader = DatasetLoader()

DOMAIN_DIR = 'dataset_aqg/output_domain/'

train_dataset = loader.load_dataset(DOMAIN_DIR, split='train')
val_dataset   = loader.load_dataset(DOMAIN_DIR, split='validation')
test_dataset  = loader.load_dataset(DOMAIN_DIR, split='test')

print(f'Train: {len(train_dataset)} | Val: {len(val_dataset)} | Test: {len(test_dataset)}')

In [ ]:
# Validate dataset
validation_results = loader.validate_dataset(train_dataset)
print(f"Duplicates: {validation_results['duplicate_count']}")
print(f"Avg input length: {validation_results['avg_input_length']} chars")

In [ ]:
# Preview sample
sample = train_dataset[0]
print('=== Sample Entry ===')
print(f"Input: {sample['input'][:200]}...")
print(f"Target: {sample['target'][:200]}...")
print(f"Format: {sample.get('metadata', {}).get('format', 'N/A')}")

## 3. Setup Model dengan LoRA

In [ ]:
from src.finetuned.model.model_setup import ModelSetup
from peft import LoraConfig

MODEL_NAME = 'LazarusNLP/IndoNanoT5-base'

model_setup = ModelSetup()

# LoRA config sesuai spec
lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    lora_dropout=0.1,
    target_modules=['q', 'v'],
    bias='none',
    task_type='SEQ_2_SEQ_LM'
)

peft_model, tokenizer = model_setup.setup_model_for_training(
    model_name=MODEL_NAME,
    lora_config=lora_config
)

In [ ]:
# Analyze token distribution
print('Analyzing token distribution...')
token_stats = loader.analyze_token_distribution(train_dataset, tokenizer, max_length=512)
print(f"Mean length: {token_stats['mean_length']} tokens")
print(f"Exceeding 512: {token_stats['pct_exceeding_limit']}%")

## 4. Configure Training

In [ ]:
from src.finetuned.training.domain_trainer import DomainAdaptationTrainer

CHECKPOINT_DIR = '/content/drive/MyDrive/AQG/checkpoints/domain'

trainer = DomainAdaptationTrainer(
    model=peft_model,
    tokenizer=tokenizer,
    output_dir=CHECKPOINT_DIR,
    max_length=512
)

# Training arguments sesuai spec
training_args = trainer.get_training_args(
    num_train_epochs=6,
    per_device_train_batch_size=8,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    warmup_steps=50,
    logging_steps=50,
    fp16=True
)

print('Training configuration ready')
print(f'Effective batch size: {training_args.per_device_train_batch_size * training_args.gradient_accumulation_steps}')

## 5. Start Training

> ⚠️ Training akan memakan waktu ~2-3 jam. Pastikan Colab tidak timeout.
> Checkpoints disimpan setiap epoch ke Google Drive.

In [ ]:
import time
start_time = time.time()

print('Starting domain adaptation training...')
print('Checkpoints will be saved to:', CHECKPOINT_DIR)
print('='*60)

results = trainer.train(
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    training_args=training_args,
    early_stopping=True,
    early_stopping_patience=2
)

elapsed = (time.time() - start_time) / 3600
print(f'\nTraining completed in {elapsed:.2f} hours')
print(f'Final training loss: {results["training_loss"]:.4f}')

## 6. Save Best Model

In [ ]:
# Save best model
model_path = trainer.save_best_model(output_name='indot5-python-domain')
print(f'Model saved to: {model_path}')

In [ ]:
# Plot training curves
trainer.plot_training_curves(
    save_path=f'{CHECKPOINT_DIR}/training_curves.png'
)

## 7. Evaluate on Validation Set

In [ ]:
from src.finetuned.evaluation.metrics_calculator import MetricsCalculator
from src.finetuned.evaluation.model_evaluator import ModelEvaluator

metrics_calc = MetricsCalculator()
evaluator = ModelEvaluator(
    model=peft_model,
    tokenizer=tokenizer,
    metrics_calculator=metrics_calc
)

# Evaluate on small subset untuk speed
print('Evaluating on validation set (first 50 samples)...')
val_metrics = evaluator.evaluate_on_test_set(
    test_dataset=val_dataset,
    num_beams=4,
    include_bertscore=False,  # Skip BERTScore untuk speed
    max_samples=50
)

In [ ]:
# Training summary
print('\n' + '='*60)
print('DOMAIN ADAPTATION TRAINING SUMMARY')
print('='*60)
print(f'Final Training Loss: {results["training_loss"]:.4f}')
print(f'Training Time: {elapsed:.2f} hours')
print(f'Model saved: {model_path}')
print(f'\nValidation Metrics:')
print(f'  BLEU-4:  {val_metrics.get("bleu_4", 0):.4f}')
print(f'  ROUGE-L: {val_metrics.get("rouge_l", 0):.4f}')
print('\n✓ Stage 1 complete! Proceed to 03_task_specific_training.ipynb')